# LOS<8 Prediction — Neural Network (MLP) Model (Lab-Only Dataset)

Simple end-to-end pipeline: load data, scale features, train a Feed-Forward Neural Network (Multi-Layer Perceptron), and evaluate it. No exploratory plots — straight to modeling and results.

**Why an MLP:** this data is tabular (numeric lab values only) with no image, text, or time-series structure, so a standard feed-forward network is the appropriate architecture here — not a CNN, RNN, or Transformer.

This version is adapted for a dataset containing **only lab-value features** (no demographic/admission fields):
`ALT, Amylase, Bilirubin, Calcium, Creatinine, Creatine Kinase, Hematocrit, Lactate, Lipase, MCH, MCHC, MCV, PT, WBC`

**Target:** `LOS<8` — 1 = length of stay < 8 days, 0 = length of stay ≥ 8 days.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

%matplotlib inline


## 1. Load data


In [ ]:
# Update this path to point to your dataset (xlsx or csv)
data_path = '../sample_datasets/MIMIC3LAB.xlsx'

if data_path.lower().endswith('.csv'):
    df = pd.read_csv(data_path)
else:
    df = pd.read_excel(data_path)

print('Shape:', df.shape)


## 2. Prepare data for modeling


In [ ]:
# No categorical columns in this dataset - use lab features directly
X = df.drop(columns=['LOS<8'])
y = df['LOS<8']

print('Feature matrix shape:', X.shape)


## 3. Train/test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)


## 4. Scale features

Neural networks (unlike tree-based models such as XGBoost/CatBoost/Random Forest) are sensitive to feature scale, so inputs are standardized to mean 0 / std 1. The scaler is fit on the training set only, then applied to both sets to avoid data leakage.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5. Train the Neural Network (MLP) model


In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

model.fit(X_train_scaled, y_train)


## 6. Evaluate the model


In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))
print('\nClassification report:\n', classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['LOS>=8', 'LOS<8'])
disp.plot(cmap='Blues')
plt.title('Confusion matrix')
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Neural Network (AUC = {auc_score:.3f})', color='#D85A30')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve')
plt.legend()
plt.show()


## 7. Feature importance

MLPs don't expose a built-in `feature_importances_` the way tree models do, so permutation importance is used instead: each feature is shuffled and the drop in ROC-AUC is measured.


In [ ]:
perm_result = permutation_importance(
    model, X_test_scaled, y_test, scoring='roc_auc', n_repeats=10, random_state=42, n_jobs=-1
)

importances = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
importances.head(15).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 15 feature importances (Neural Network, permutation-based)')
plt.xlabel('Mean decrease in ROC-AUC')
plt.show()


## 8. Results summary (for your report)


In [ ]:
print('=== Model Results Summary ===')
print('Model: Neural Network (MLP Classifier)')
print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}')

report_dict = classification_report(y_test, y_pred, output_dict=True)
print(f"Precision (LOS<8=1): {report_dict['1']['precision']:.3f}")
print(f"Recall (LOS<8=1): {report_dict['1']['recall']:.3f}")
print(f"F1-score (LOS<8=1): {report_dict['1']['f1-score']:.3f}")
